# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [40]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.540     0.333     0.412      9389
           1      0.683     0.835     0.751     16162

    accuracy                          0.651     25551
   macro avg      0.611     0.584     0.582     25551
weighted avg      0.630     0.651     0.627     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [42]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import classification_report

# ===== Your turn: 90-day window + new feature + client-level split =====
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last45,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev45,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last45,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END)       AS pos_last45,
               STDDEV(f.gsc_avg_position) AS pos_volatility
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev45 >= 50
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features_90d):,} content items with enough history (90-day window)')

data_90d = features_90d.merge(qsignals, on='content_hash_id', how='left')
data_90d['is_declining'] = (data_90d['imp_last45'] < 0.8 * data_90d['imp_prev45']).astype(int)

feature_cols_90d = ['imp_prev45', 'pos_volatility', 'visible_queries',
                     'rare_share', 'anon_share', 'top_query_share']
model_data_90d = data_90d.dropna(subset=feature_cols_90d)
print(f'{len(model_data_90d):,} rows ready for modeling')

X = model_data_90d[feature_cols_90d]
y = model_data_90d['is_declining']
groups = model_data_90d['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model_grouped.predict(X_te), digits=3))

print("\n=== Feature importance ===")
for feat, imp in sorted(zip(feature_cols_90d, model_grouped.feature_importances_), key=lambda x: -x[1]):
    print(f'{feat:20} {imp:.3f}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

142,816 content items with enough history (90-day window)
116,676 rows ready for modeling
base rate (always predict majority): 0.669
              precision    recall  f1-score   support

           0      0.811     0.653     0.724      4979
           1      0.844     0.925     0.882     10065

    accuracy                          0.835     15044
   macro avg      0.827     0.789     0.803     15044
weighted avg      0.833     0.835     0.830     15044


=== Feature importance ===
imp_prev45           0.230
pos_volatility       0.186
anon_share           0.171
rare_share           0.160
top_query_share      0.127
visible_queries      0.126


In [43]:
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model_random = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_r, y_tr_r)

print("=== RANDOM SPLIT (90-day features) ===")
print(f'base rate: {max(y_te_r.mean(), 1 - y_te_r.mean()):.3f}')
print(classification_report(y_te_r, model_random.predict(X_te_r), digits=3))

print("\n=== GROUPED SPLIT (90-day features) — already have this ===")
print(f'base rate: {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model_grouped.predict(X_te), digits=3))

=== RANDOM SPLIT (90-day features) ===
base rate: 0.617
              precision    recall  f1-score   support

           0      0.759     0.629     0.688     11172
           1      0.792     0.876     0.832     17997

    accuracy                          0.781     29169
   macro avg      0.775     0.753     0.760     29169
weighted avg      0.779     0.781     0.777     29169


=== GROUPED SPLIT (90-day features) — already have this ===
base rate: 0.669
              precision    recall  f1-score   support

           0      0.811     0.653     0.724      4979
           1      0.844     0.925     0.882     10065

    accuracy                          0.835     15044
   macro avg      0.827     0.789     0.803     15044
weighted avg      0.833     0.835     0.830     15044



In [44]:
feature_cols_no_vol = ['imp_prev45', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']

X_no_vol = model_data_90d[feature_cols_no_vol]
X_tr_nv, X_te_nv = X_no_vol.iloc[train_idx], X_no_vol.iloc[test_idx]
y_tr_nv, y_te_nv = y.iloc[train_idx], y.iloc[test_idx]

model_no_vol = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr_nv, y_tr_nv)

print("=== GROUPED SPLIT, WITHOUT pos_volatility ===")
print(classification_report(y_te_nv, model_no_vol.predict(X_te_nv), digits=3))

print("\n=== GROUPED SPLIT, WITH pos_volatility ===")
print(classification_report(y_te, model_grouped.predict(X_te), digits=3))

=== GROUPED SPLIT, WITHOUT pos_volatility ===
              precision    recall  f1-score   support

           0      0.765     0.671     0.715      4979
           1      0.847     0.898     0.872     10065

    accuracy                          0.823     15044
   macro avg      0.806     0.785     0.793     15044
weighted avg      0.820     0.823     0.820     15044


=== GROUPED SPLIT, WITH pos_volatility ===
              precision    recall  f1-score   support

           0      0.811     0.653     0.724      4979
           1      0.844     0.925     0.882     10065

    accuracy                          0.835     15044
   macro avg      0.827     0.789     0.803     15044
weighted avg      0.833     0.835     0.830     15044



In [45]:
proba = model_grouped.predict_proba(X_te)[:, 1]
top50_idx = np.argsort(proba)[-50:]
precision_at_50_model = y_te.iloc[top50_idx].mean()
print(f"Precision@50 (model):    {precision_at_50_model:.3f}")

baseline_rank = X_te['imp_prev45'].to_numpy()
baseline_top50_idx = np.argsort(baseline_rank)[-50:]
precision_at_50_baseline = y_te.iloc[baseline_top50_idx].mean()
print(f"Precision@50 (baseline): {precision_at_50_baseline:.3f}")

Precision@50 (model):    1.000
Precision@50 (baseline): 0.540


In [46]:
proba = model_grouped.predict_proba(X_te)[:, 1]
top50_idx = np.argsort(proba)[-50:]
print(proba[top50_idx][:10])
print(len(set(top50_idx)))

[0.995 0.995 0.995 0.995 0.995 0.995 0.995 0.995 0.995 0.995]
50


In [47]:
import pandas as pd

# Get probability of decline for every row in the test set
test_ids = model_data_90d.iloc[test_idx][['client_hash_id', 'content_hash_id', 'imp_prev45']].reset_index(drop=True)
test_ids['decline_proba'] = model_grouped.predict_proba(X_te)[:, 1]

# Define thresholds — median split on value, and a reasonable probability cutoff
value_median = test_ids['imp_prev45'].median()
proba_high_cutoff = 0.6  # tune this later against your Precision@50 finding

def assign_action(row):
    high_value = row['imp_prev45'] >= value_median
    high_risk = row['decline_proba'] >= proba_high_cutoff
    if high_risk and high_value:
        return "Refresh urgently"
    elif high_risk and not high_value:
        return "Monitor"
    elif not high_risk and high_value:
        return "Protect"
    else:
        return "Prune/merge candidate"

test_ids['recommended_action'] = test_ids.apply(assign_action, axis=1)

print(test_ids['recommended_action'].value_counts())
test_ids.sort_values('decline_proba', ascending=False).head(20)

recommended_action
Refresh urgently         5411
Monitor                  4554
Prune/merge candidate    2964
Protect                  2115
Name: count, dtype: int64


,client_hash_id,content_hash_id,imp_prev45,decline_proba,recommended_action
4383,client_e5c2aa26a8598242,content_84c12684ff9a5595,1178.0,1.0,Refresh urgently
1185,client_fef1a8f436438636,content_8633ee0bda8a2e2b,6419.0,1.0,Refresh urgently
8140,client_fef1a8f436438636,content_9d97271cf40958c4,1360.0,1.0,Refresh urgently
11332,client_e5c2aa26a8598242,content_cae2ce6c87a2de4c,1139.0,1.0,Refresh urgently
3259,client_e5c2aa26a8598242,content_04d45973df6684d8,2089.0,1.0,Refresh urgently
8072,client_fef1a8f436438636,content_1919e13e207ee383,2899.0,1.0,Refresh urgently
10785,client_e5c2aa26a8598242,content_fe8060a9d94e95d0,1904.0,1.0,Refresh urgently
5068,client_fef1a8f436438636,content_9962ba19b45f0c4b,1711.0,1.0,Refresh urgently
8878,client_fef1a8f436438636,content_8d72d9f24003a1de,451.0,1.0,Monitor
12879,client_fef1a8f436438636,content_1fb1b0da2e27fd47,480.0,1.0,Monitor


In [48]:
test_ids['client_hash_id'].nunique()

13